# `qwen3-4b-4bit-qlora-s00-r0` — DGX Spark recipe

Container-aware. Run inside `nvcr.io/nvidia/pytorch:25.11-py3` (launched by `models/recipes/dgx_spark/launch.sh`).
Substrate-of-record: NVIDIA DGX Spark Unsloth playbook. Hardware notes: `.meta/hardware.md`.

**Decision log** (Leo, 2026-05-20):
- **Supersession-note rendering**: DEFERRED to r1. Diana's `_format.py` does not template-render `meta.gaap_supersession` into the assistant turn, AND the rendered split JSONL does not carry the supersession block in its slim `meta`. Re-rendering requires regenerating the splits. r0 trains on raw Spiceland gold; the drift is a post-hoc Vera eval column.
- **Loss weighting 0.85/0.15**: DEFERRED to r1. r0 trains uniformly (`SFTTrainer` default 1.0/1.0). Custom per-segment weighting requires a custom collator; not worth blocking r0.
- **r0 launch trigger**: user-gated. This notebook is *prepared*, not *executed*.

## §1 — Substrate guardrails

Fail fast if we are on the wrong host, wrong CUDA arch, wrong split version, or the seed split sha drifted from `eval/sft/splits/manifest.json`.

In [ ]:
import hashlib, json, os, sys
from pathlib import Path

REPO = Path("/workspace")
SEED_DIR = REPO / "eval/sft/splits/seed_00__351199285"
MANIFEST_PATH = REPO / "eval/sft/splits/manifest.json"
RUN_ID = "qwen3-4b-4bit-qlora-s00-r0"
RUN_DIR = REPO / f"models/runs/{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

import torch
assert torch.cuda.is_available(), "CUDA not visible — launch via launch.sh with --gpus all"
cc = torch.cuda.get_device_capability(0)
name = torch.cuda.get_device_name(0)
print(f"device: {name}  compute_cap={cc[0]}.{cc[1]}")
assert cc == (12, 1), f"expected GB10 sm_120 (12,1), got {cc}"
assert "GB10" in name, f"expected GB10 device, got {name!r}"

for split in ("train", "valid", "test"):
    p = SEED_DIR / f"{split}.jsonl"
    assert p.exists(), f"missing {p}"

manifest = json.loads(MANIFEST_PATH.read_text())
assert manifest["source_jsonl_sha256"].startswith("fed6eb17de8be1e4"), (
    f"corpus drift: manifest source sha = {manifest['source_jsonl_sha256'][:16]}, "
    "expected spiceland9e-v1.1.0 = fed6eb17de8be1e4\u2026"
)
seed00 = next(s for s in manifest["seeds"] if s["seed_index"] == 0)
for split, want in seed00["file_sha256"].items():
    got = hashlib.sha256((SEED_DIR / f"{split}.jsonl").read_bytes()).hexdigest()
    assert got == want, f"{split} sha drift: got {got[:16]}\u2026 want {want[:16]}\u2026"
print("seed_00 splits sha-verified against manifest:")
for split, want in seed00["file_sha256"].items():
    print(f"  {split:<5} {want[:16]}\u2026")
print("corpus anchor: spiceland9e-v1.1.0")
print(f"counts: {seed00['split_counts']}")

## §2 — Dataset + tokenizer

Diana's splits already carry rendered `messages` (system + user + assistant) per `eval/_format.py`. We map those to a single `text` field via the Qwen3 chat template. `meta.exclude_from_scoring` is already filtered upstream at split time (manifest reports 1 skip). The supersession-note rendering is **NOT** present in the split JSONL — see decision log above; r0 trains on raw Spiceland gold.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

BASE_MODEL_PRIMARY = "unsloth/Qwen3-4B-Instruct-2507"
BASE_MODEL_FALLBACK = "Qwen/Qwen3-4B-Instruct-2507"
MAX_SEQ_LEN = 2048

tok_probe = AutoTokenizer.from_pretrained(BASE_MODEL_PRIMARY, trust_remote_code=False)
TOKENIZER_REVISION = getattr(tok_probe, "name_or_path", BASE_MODEL_PRIMARY)
print(f"tokenizer probe ok: {TOKENIZER_REVISION}")
print(f"chat template present: {bool(tok_probe.chat_template)}")

ds = load_dataset(
    "json",
    data_files={
        "train": str(SEED_DIR / "train.jsonl"),
        "valid": str(SEED_DIR / "valid.jsonl"),
        "test":  str(SEED_DIR / "test.jsonl"),
    },
)
print({k: len(v) for k, v in ds.items()})

def render_text(rec):
    # rec["messages"] = [{role, content}, ...]; apply Qwen3 chat template with
    # the assistant turn fully present so train_on_responses_only can mask the
    # prompt half. Do not add a generation prompt.
    return tok_probe.apply_chat_template(
        rec["messages"], tokenize=False, add_generation_prompt=False
    )

ds = ds.map(lambda r: {"text": render_text(r)}, num_proc=4)
print("sample (first 600 chars of train[0].text):")
print(ds["train"][0]["text"][:600])

## §3 — Base model (4-bit QLoRA via Unsloth `FastModel`)

Playbook-validated path: `FastModel.from_pretrained(load_in_4bit=True, full_finetuning=False)`. Try the Unsloth pre-quantized repo first; on miss, fall back to the official Qwen repo and let Unsloth auto-quantize.

In [ ]:
from unsloth import FastModel, FastLanguageModel

BASE_MODEL_USED = BASE_MODEL_PRIMARY
try:
    model, tokenizer = FastModel.from_pretrained(
        model_name=BASE_MODEL_PRIMARY,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
        load_in_8bit=False,
        full_finetuning=False,
    )
except Exception as e:
    print(f"primary load failed ({type(e).__name__}: {e!s:.200}); falling back to {BASE_MODEL_FALLBACK}")
    BASE_MODEL_USED = BASE_MODEL_FALLBACK
    model, tokenizer = FastModel.from_pretrained(
        model_name=BASE_MODEL_FALLBACK,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
        load_in_8bit=False,
        full_finetuning=False,
    )
TOKENIZER_REVISION = getattr(tokenizer, "name_or_path", BASE_MODEL_USED)
print(f"base loaded: {BASE_MODEL_USED}")
print(f"tokenizer:   {TOKENIZER_REVISION}")

## §4 — LoRA adapter

Rank 16 chosen over the default 8 because the corpus has structured-output content (journal entries, multi-step rationales) that benefits from extra adapter capacity. `lora_alpha=32` keeps the alpha/r ratio at 2.0 (Unsloth's common ratio for QLoRA SFT); the playbook reference uses 16/16 but on small-corpus instruct tuning 2:1 is more stable.

In [ ]:
SEED = 351199285
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=TARGET_MODULES,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

## §5 — SFTTrainer config

Half-epoch eval (`eval_steps=110`): with 3,538 train rows / (batch 2 × grad-accum 4) ≈ 442 steps/epoch × 3 ≈ 1,326 total steps; 110 ≈ quarter-epoch, so we get ≈12 dev-loss readings to feed EarlyStoppingCallback (patience=3).

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

OUTPUT_DIR = str(RUN_DIR)

sft_cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.05,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    weight_decay=0.0,
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    dataset_text_field="text",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=110,
    save_strategy="steps",
    save_steps=110,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    fp16=False,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["valid"],
    args=sft_cfg,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

## §6 — `train_on_responses_only` masking

Mask the prompt half so the loss is computed only on the assistant turn. Qwen3 instruction templates are `<|im_start|>user` ... `<|im_start|>assistant` ... `<|im_end|>`.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
print("responses-only masking applied (Qwen3 im_start/im_end tags)")

## §7 — Train

Smoke-test projection (Phi-3.5-mini @ 4.30 samples/s on this host): Qwen3-4B at ≈ same throughput → 3,538 train rows × 3 epochs / 4.3 ≈ 41 min wall-clock for r0.

In [ ]:
import time
t0 = time.time()
train_result = trainer.train()
wall_clock_s = time.time() - t0
print(f"train done in {wall_clock_s/60:.1f} min")
print(train_result.metrics)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"adapter + tokenizer saved \u2192 {OUTPUT_DIR}")

## §8 — Manifest emission

Per Leo's standing rule — every run pinned. The `source_jsonl_sha256` binds this adapter to `spiceland9e-v1.1.0`.

In [ ]:
import datetime as dt

def file_sha256(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

adapter_path = RUN_DIR / "adapter_model.safetensors"
if not adapter_path.exists():
    candidates = list(RUN_DIR.glob("adapter_model*"))
    adapter_path = candidates[0] if candidates else None
adapter_sha = file_sha256(adapter_path) if adapter_path and adapter_path.exists() else None

best_dev_loss = None
log_history = getattr(trainer.state, "log_history", [])
for entry in log_history:
    if "eval_loss" in entry:
        v = entry["eval_loss"]
        if best_dev_loss is None or v < best_dev_loss:
            best_dev_loss = v

train_samples = len(ds["train"])
throughput = (train_result.metrics.get("train_samples_per_second")
              if train_result and train_result.metrics else None)

manifest_out = {
    "run_id": RUN_ID,
    "created_at": dt.datetime.utcnow().isoformat() + "Z",
    "base_model": BASE_MODEL_USED,
    "tokenizer_revision": TOKENIZER_REVISION,
    "adapter_sha256": adapter_sha,
    "source_jsonl_sha256": manifest["source_jsonl_sha256"],
    "corpus_anchor": "spiceland9e-v1.1.0",
    "train_split_sha256": seed00["file_sha256"]["train"],
    "valid_split_sha256": seed00["file_sha256"]["valid"],
    "holdout_split_sha256": seed00["file_sha256"]["test"],
    "seed_int": SEED,
    "recipe": {
        "method": "qlora-4bit",
        "rank": 16,
        "alpha": 32,
        "dropout": 0,
        "target_modules": TARGET_MODULES,
        "max_seq_length": MAX_SEQ_LEN,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-4,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.05,
        "num_train_epochs": 3,
        "optim": "adamw_8bit",
        "bf16": True,
        "loss_weighting": "uniform",
        "loss_weighting_note": (
            "r0 trains uniformly. Pat's 0.85/0.15 gold/supersession weighting deferred to r1; "
            "requires custom collator + supersession block plumbed into split records."
        ),
        "supersession_render": "absent",
        "supersession_note": (
            "r0 trains on raw Spiceland gold. eval/_format.py does not render "
            "meta.gaap_supersession into the assistant turn, and Diana's split JSONL drops "
            "the field. Deferred to r1."
        ),
        "responses_only_masking": True,
    },
    "final_dev_loss": best_dev_loss,
    "final_train_loss": train_result.metrics.get("train_loss") if train_result else None,
    "wall_clock_seconds": wall_clock_s,
    "throughput_samples_per_second": throughput,
    "train_samples": train_samples,
    "gpu_device": name,
    "cuda_compute_cap": f"{cc[0]}.{cc[1]}",
    "container_image": "nvcr.io/nvidia/pytorch:25.11-py3",
    "unsloth_version": "2026.5.5",
    "trl_version": "0.26.1",
    "datasets_version": "4.3.0",
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest_out, indent=2) + "\n")
print(json.dumps(manifest_out, indent=2))

## §9 — Quick eval (50-sample probe)

NOT production eval. Vera's `eval/run.py` is the production path (pending). This is a sanity-check that the adapter generates sensible completions before the user gates a full Vera run.

In [ ]:
import random
FastLanguageModel.for_inference(model)

rng = random.Random(SEED)
test_rows = list(ds["test"])
probe = rng.sample(test_rows, k=min(50, len(test_rows)))
probe_path = RUN_DIR / "probe_predictions.jsonl"

with probe_path.open("w") as fh:
    for i, rec in enumerate(probe):
        msgs = rec["messages"][:2]
        prompt_text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        out = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
        completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        row = {
            "id": rec["meta"]["id"],
            "chapter": rec["meta"]["chapter"],
            "type": rec["meta"]["type"],
            "primary_bloom": rec["meta"]["primary_bloom"],
            "gold_assistant": rec["messages"][2]["content"],
            "prediction": completion,
        }
        fh.write(json.dumps(row) + "\n")
        if i < 3:
            print(f"--- probe {i} [{row['id']}, {row['type']}] ---")
            print(f"GOLD : {row['gold_assistant'][:200]}")
            print(f"PRED : {row['prediction'][:200]}")
print(f"probe predictions \u2192 {probe_path}")

## §10 — GGUF export (optional, gated)

For Ollama / llama.cpp portability. Disabled by default; flip the flag if a portable artifact is wanted post-Vera-PASS.

In [ ]:
do_gguf_export = False

if do_gguf_export:
    gguf_dir = RUN_DIR / "gguf"
    gguf_dir.mkdir(exist_ok=True)
    model.save_pretrained_gguf(
        str(gguf_dir),
        tokenizer,
        quantization_method="q4_k_m",
    )
    print(f"GGUF q4_k_m \u2192 {gguf_dir}")
else:
    print("GGUF export skipped (flag off). Enable by setting do_gguf_export=True.")